# Hyperparameter Tuning & Model Optimization

## Objective

The objective of this notebook is to optimize the top-performing models identified during the baseline model training phase.

The following models are selected for tuning:

- LightGBM
- XGBoost
- CatBoost

The optimized models will be compared against their baseline versions using:

- Recall
- F1 Score
- ROC-AUC

The best-performing model will be selected as the final production model.

In [6]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## Load Training and Testing Artifacts

In [2]:
train_df = pd.read_csv("../artifacts/train.csv")
test_df = pd.read_csv("../artifacts/test.csv")

preprocessor = joblib.load(
    "../artifacts/preprocessor.pkl"
)

In [3]:
X_train = train_df.drop("Diagnosis", axis=1)
y_train = train_df["Diagnosis"]

X_test = test_df.drop("Diagnosis", axis=1)
y_test = test_df["Diagnosis"]

In [4]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Baseline Cross Validation

Cross validation is performed to evaluate model stability across multiple training folds.

In [7]:
lgbm = LGBMClassifier(random_state=42)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    lgbm,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("Mean ROC AUC:", scores.mean())

[LightGBM] [Info] Number of positive: 487, number of negative: 888
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000594 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3290
[LightGBM] [Info] Number of data points in the train set: 1375, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.354182 -> initscore=-0.600708
[LightGBM] [Info] Start training from score -0.600708
[LightGBM] [Info] Number of positive: 486, number of negative: 889
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000685 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3290
[LightGBM] [Info] Number of data points in the train set: 1375, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.353455 -> initscore=-0.603889
[LightGBM] [Info] Start training from score -0.603889
[LightGBM] [Info] Numb

In [9]:
lgbm_params = {
    "n_estimators":[100,200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5,7,10],
    "num_leaves":[15,31,63],
    "subsample":[0.7,0.8,1.0]
}

In [10]:
lgbm_search = RandomizedSearchCV(
    estimator=LGBMClassifier(
        random_state=42
    ),
    param_distributions=lgbm_params,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

lgbm_search.fit(
    X_train_processed,
    y_train
)

[LightGBM] [Info] Number of positive: 608, number of negative: 1111
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3290
[LightGBM] [Info] Number of data points in the train set: 1719, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.353694 -> initscore=-0.602841
[LightGBM] [Info] Start training from score -0.602841
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], 'num_leaves': [15, 31, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for 

In [11]:
print(lgbm_search.best_params_)

{'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}


In [12]:
best_lgbm = lgbm_search.best_estimator_

lgbm_pred = best_lgbm.predict(
    X_test_processed
)

lgbm_prob = best_lgbm.predict_proba(
    X_test_processed
)[:,1]

In [13]:
tuned_results = []
tuned_results.append({
    "Model":"LightGBM",
    "Accuracy":accuracy_score(y_test,lgbm_pred),
    "Precision":precision_score(y_test,lgbm_pred),
    "Recall":recall_score(y_test,lgbm_pred),
    "F1":f1_score(y_test,lgbm_pred),
    "ROC_AUC":roc_auc_score(y_test,lgbm_prob)
})

In [14]:
xgb = XGBClassifier(random_state=42)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    xgb,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("Mean ROC AUC:", scores.mean())

Mean ROC AUC: 0.9563358125687291


In [16]:
xgb_params = {
    "n_estimators":[100,200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5,7],
    "subsample":[0.7,0.8,1.0],
    "colsample_bytree":[0.7,0.8,1.0]
}

In [18]:
xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=42
    ),
    param_distributions=xgb_params,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(
    X_train_processed,
    y_train
)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance

In [20]:
print(xgb_search.best_params_)

{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


In [23]:
best_xgb = xgb_search.best_estimator_

xgb_pred = best_xgb.predict(
    X_test_processed
)

xgb_prob = best_xgb.predict_proba(
    X_test_processed
)[:,1]

In [24]:
tuned_results.append({
    "Model":"XGBoost",
    "Accuracy":accuracy_score(y_test,xgb_pred),
    "Precision":precision_score(y_test,xgb_pred),
    "Recall":recall_score(y_test,xgb_pred),
    "F1":f1_score(y_test,xgb_pred),
    "ROC_AUC":roc_auc_score(y_test,xgb_prob)
})

In [15]:
catboost = CatBoostClassifier(random_state=42)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    catboost,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("Mean ROC AUC:", scores.mean())

Learning rate set to 0.011803
0:	learn: 0.6780314	total: 160ms	remaining: 2m 39s
1:	learn: 0.6650904	total: 164ms	remaining: 1m 21s
2:	learn: 0.6529170	total: 169ms	remaining: 56.1s
3:	learn: 0.6403083	total: 175ms	remaining: 43.6s
4:	learn: 0.6266576	total: 178ms	remaining: 35.5s
5:	learn: 0.6138506	total: 182ms	remaining: 30.2s
6:	learn: 0.6007866	total: 185ms	remaining: 26.3s
7:	learn: 0.5900311	total: 189ms	remaining: 23.4s
8:	learn: 0.5805905	total: 192ms	remaining: 21.1s
9:	learn: 0.5675199	total: 194ms	remaining: 19.3s
10:	learn: 0.5560367	total: 201ms	remaining: 18.1s
11:	learn: 0.5454849	total: 205ms	remaining: 16.9s
12:	learn: 0.5347772	total: 208ms	remaining: 15.8s
13:	learn: 0.5251495	total: 210ms	remaining: 14.8s
14:	learn: 0.5147270	total: 214ms	remaining: 14s
15:	learn: 0.5039426	total: 218ms	remaining: 13.4s
16:	learn: 0.4934598	total: 221ms	remaining: 12.8s
17:	learn: 0.4838078	total: 225ms	remaining: 12.3s
18:	learn: 0.4746786	total: 230ms	remaining: 11.9s
19:	learn: 

In [17]:
cat_params = {
    "iterations":[100,200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "depth":[3,5,7,10]
}

In [19]:
cb_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        random_state=42
    ),
    param_distributions=cat_params,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

cb_search.fit(
    X_train_processed,
    y_train
)

0:	learn: 0.6294728	total: 51.2ms	remaining: 15.3s
1:	learn: 0.5766226	total: 101ms	remaining: 15s
2:	learn: 0.5373447	total: 150ms	remaining: 14.8s
3:	learn: 0.4961415	total: 196ms	remaining: 14.5s
4:	learn: 0.4602273	total: 246ms	remaining: 14.5s
5:	learn: 0.4272765	total: 298ms	remaining: 14.6s
6:	learn: 0.3933806	total: 347ms	remaining: 14.5s
7:	learn: 0.3657646	total: 360ms	remaining: 13.1s
8:	learn: 0.3442599	total: 410ms	remaining: 13.3s
9:	learn: 0.3236867	total: 461ms	remaining: 13.4s
10:	learn: 0.3089674	total: 514ms	remaining: 13.5s
11:	learn: 0.2931925	total: 563ms	remaining: 13.5s
12:	learn: 0.2774586	total: 617ms	remaining: 13.6s
13:	learn: 0.2638732	total: 664ms	remaining: 13.6s
14:	learn: 0.2511613	total: 716ms	remaining: 13.6s
15:	learn: 0.2409347	total: 766ms	remaining: 13.6s
16:	learn: 0.2311577	total: 816ms	remaining: 13.6s
17:	learn: 0.2219079	total: 866ms	remaining: 13.6s
18:	learn: 0.2138312	total: 913ms	remaining: 13.5s
19:	learn: 0.2054384	total: 959ms	remainin

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",CatBoostClass...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'depth': [3, 5, ...], 'iterations': [100, 200, ...], 'learning_rate': [0.01, 0.05, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attribut

In [21]:
print(cb_search.best_params_)

{'learning_rate': 0.05, 'iterations': 300, 'depth': 10}


In [22]:
best_cb = cb_search.best_estimator_

cb_pred = best_cb.predict(
    X_test_processed
)

cb_prob = best_cb.predict_proba(
    X_test_processed
)[:,1]

In [25]:
tuned_results.append({
    "Model":"CatBoost",
    "Accuracy":accuracy_score(y_test,cb_pred),
    "Precision":precision_score(y_test,cb_pred),
    "Recall":recall_score(y_test,cb_pred),
    "F1":f1_score(y_test,cb_pred),
    "ROC_AUC":roc_auc_score(y_test,cb_prob)
})

In [26]:
tuned_results_df = pd.DataFrame(
    tuned_results
)

tuned_results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,LightGBM,0.951163,0.945578,0.914474,0.929766,0.952149
2,CatBoost,0.953488,0.945946,0.921053,0.933333,0.944789
1,XGBoost,0.946512,0.938776,0.907895,0.923077,0.943653


## Hyperparameter Tuning Observation

Hyperparameter tuning resulted in a marginal improvement in ROC-AUC score. However, the baseline LightGBM model achieved higher Recall and F1 Score.

Since Alzheimer's disease prediction prioritizes the identification of positive cases, Recall is considered the primary evaluation metric.

Therefore, the baseline LightGBM model is selected as the final model for deployment.

In [27]:
import joblib

final_model = LGBMClassifier(
    random_state=42
)

final_model.fit(
    X_train_processed,
    y_train
)

joblib.dump(
    final_model,
    "../artifacts/model.pkl"
)

[LightGBM] [Info] Number of positive: 608, number of negative: 1111
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000615 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3290
[LightGBM] [Info] Number of data points in the train set: 1719, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.353694 -> initscore=-0.602841
[LightGBM] [Info] Start training from score -0.602841


['../artifacts/model.pkl']